# Wall hit statistics

This notebook rolls out the selected variant-2 models on the test episodes and compares how often each policy attempts an invalid movement.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch


NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "statistic" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import rainbow_dqn
from environment_v5 import Environment_v5
from environment_v13 import Environment_v13


DATA_DIR = PROJECT_ROOT / "data"
MODEL_ROOT = PROJECT_ROOT / "models2"
VARIANT = 2
NUM_TEST_EPISODES = 100

ACTION_NAMES = {
    0: "stay",
    1: "up",
    2: "right",
    3: "down",
    4: "left",
}

RUNS = [
    {
        "name": "DQN v8.13.2 + env13",
        "short_name": "v8.13.2",
        "env_class": Environment_v13,
        "model_path": MODEL_ROOT / "DQN_v8.13.2_variant_2.pt",
        "color": "tab:blue",
    },
    {
        "name": "DQN v8.5.31 + env5",
        "short_name": "v8.5.31",
        "env_class": Environment_v5,
        "model_path": MODEL_ROOT / "DQN_v8.5.31_variant_2.pt",
        "color": "tab:orange",
    },
]


print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {DATA_DIR}")
print(f"Model root: {MODEL_ROOT}")

In [ ]:
def intended_location(loc, action):
    row, col = loc
    if action == 1:
        return (row - 1, col)
    if action == 2:
        return (row, col + 1)
    if action == 3:
        return (row + 1, col)
    if action == 4:
        return (row, col - 1)
    return loc


def invalid_move_kind(env, intended_loc):
    row, col = intended_loc
    if row < 0 or row >= env.vertical_cell_count or col < 0 or col >= env.horizontal_cell_count:
        return "boundary"
    if intended_loc not in env.eligible_cells:
        return "blocked_cell"
    return "unknown"


def load_policy(env, model_path):
    network = rainbow_dqn.DQN_v8(env)
    network.device = torch.device("cpu")
    network.support = network.support.to(network.device)
    network.q_network.to(network.device)
    network.target_network.to(network.device)

    state_dict = torch.load(model_path, map_location=network.device)
    network.q_network.load_state_dict(state_dict)
    network.q_network.eval()
    network.epsilon = 0
    return network


def rollout_wall_hits(run, episode_ids):
    env = run["env_class"](variant=VARIANT, data_dir=str(DATA_DIR))
    policy = load_policy(env, run["model_path"])
    episode_rows = []
    hit_rows = []

    for episode_id in episode_ids:
        env.test_episodes = [int(episode_id)]
        obs = env.reset("testing")

        total_reward = 0.0
        wall_hits = 0
        boundary_hits = 0
        blocked_cell_hits = 0
        wall_hits_at_target = 0
        wall_hits_elsewhere = 0
        stay_actions = 0
        move_actions = 0

        for step in range(1, env.episode_steps + 1):
            before_loc = env.agent_loc
            action = int(policy.select_action(obs))

            if action == 0:
                stay_actions += 1
            else:
                move_actions += 1

            target_loc = intended_location(before_loc, action)
            reward, next_obs, done = env.step(action)
            after_loc = env.agent_loc
            total_reward += reward

            if action != 0 and after_loc == before_loc:
                wall_hits += 1
                kind = invalid_move_kind(env, target_loc)
                boundary_hits += kind == "boundary"
                blocked_cell_hits += kind == "blocked_cell"
                wall_hits_at_target += before_loc == env.target_loc
                wall_hits_elsewhere += before_loc != env.target_loc
                hit_rows.append({
                    "model": run["name"],
                    "short_name": run["short_name"],
                    "episode_id": int(episode_id),
                    "step": step,
                    "from_row": before_loc[0],
                    "from_col": before_loc[1],
                    "intended_row": target_loc[0],
                    "intended_col": target_loc[1],
                    "action": action,
                    "action_name": ACTION_NAMES[action],
                    "kind": kind,
                    "at_target": before_loc == env.target_loc,
                })

            obs = next_obs
            if done:
                break

        episode_rows.append({
            "model": run["name"],
            "short_name": run["short_name"],
            "episode_id": int(episode_id),
            "test_episode": int(episode_id) + 1,
            "total_reward": total_reward,
            "wall_hits": wall_hits,
            "boundary_hits": boundary_hits,
            "blocked_cell_hits": blocked_cell_hits,
            "wall_hits_at_target": wall_hits_at_target,
            "wall_hits_elsewhere": wall_hits_elsewhere,
            "stay_actions": stay_actions,
            "move_actions": move_actions,
        })

    return pd.DataFrame(episode_rows), pd.DataFrame(hit_rows)


episode_ids = pd.read_csv(DATA_DIR / f"variant_{VARIANT}" / "test_episodes.csv")["test_episodes"].head(NUM_TEST_EPISODES)
episode_frames = []
hit_frames = []

for run in RUNS:
    print(f"Rolling out {run['name']}...")
    episode_df, hit_df = rollout_wall_hits(run, episode_ids)
    episode_frames.append(episode_df)
    hit_frames.append(hit_df)

wall_hit_by_episode = pd.concat(episode_frames, ignore_index=True)
wall_hit_events = pd.concat(hit_frames, ignore_index=True)

summary = wall_hit_by_episode.groupby(["model", "short_name"], as_index=False).agg(
    episodes=("episode_id", "count"),
    total_reward=("total_reward", "sum"),
    average_reward=("total_reward", "mean"),
    total_wall_hits=("wall_hits", "sum"),
    average_wall_hits=("wall_hits", "mean"),
    median_wall_hits=("wall_hits", "median"),
    total_boundary_hits=("boundary_hits", "sum"),
    total_blocked_cell_hits=("blocked_cell_hits", "sum"),
    total_wall_hits_at_target=("wall_hits_at_target", "sum"),
    total_wall_hits_elsewhere=("wall_hits_elsewhere", "sum"),
)

summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.8), dpi=140)
colors = {run["name"]: run["color"] for run in RUNS}
run_order = [run["name"] for run in RUNS]
short_labels = [run["short_name"] for run in RUNS]

totals = summary.set_index("model").loc[run_order]
axes[0].bar(
    short_labels,
    totals["total_wall_hits"],
    color=[colors[name] for name in run_order],
    alpha=0.88,
)
for idx, value in enumerate(totals["total_wall_hits"]):
    axes[0].text(idx, value, f"{int(value)}", ha="center", va="bottom", fontsize=10)
axes[0].set_title("Total wall hits", loc="left", pad=12)
axes[0].set_ylabel("Wall hits", labelpad=10)
axes[0].grid(axis="y", linestyle="-", alpha=0.18)

box_data = [
    wall_hit_by_episode.loc[wall_hit_by_episode["model"] == name, "wall_hits"]
    for name in run_order
]
box = axes[1].boxplot(box_data, tick_labels=short_labels, patch_artist=True, widths=0.5)
for patch, name in zip(box["boxes"], run_order):
    patch.set_facecolor(colors[name])
    patch.set_alpha(0.35)
for median in box["medians"]:
    median.set_color("#222222")
    median.set_linewidth(1.6)
axes[1].set_title("Wall hits per test episode", loc="left", pad=12)
axes[1].set_ylabel("Wall hits", labelpad=10)
axes[1].grid(axis="y", linestyle="-", alpha=0.18)

info = (
    f"Variant: {VARIANT}\n"
    f"Test episodes: {NUM_TEST_EPISODES}\n"
    "Wall hit: action != stay and location unchanged"
)
axes[1].text(
    0.98,
    0.98,
    info,
    transform=axes[1].transAxes,
    ha="right",
    va="top",
    bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "edgecolor": "#cccccc", "alpha": 0.92},
    fontsize=9,
)

plt.tight_layout()
plt.show()

summary

In [ ]:
paired = wall_hit_by_episode.pivot(index="episode_id", columns="short_name", values="wall_hits").reset_index()
paired["diff_v8.13.2_minus_v8.5.31"] = paired["v8.13.2"] - paired["v8.5.31"]

fig, ax = plt.subplots(figsize=(12, 5.8), dpi=140)
ax.axhline(0, color="#333333", linewidth=1.0, alpha=0.75)
ax.bar(
    paired["episode_id"] + 1,
    paired["diff_v8.13.2_minus_v8.5.31"],
    color=np.where(paired["diff_v8.13.2_minus_v8.5.31"] <= 0, "tab:blue", "tab:orange"),
    alpha=0.82,
    width=0.8,
)
ax.set_title("Per-episode wall hit difference", loc="left", pad=12)
ax.set_xlabel("Test episode", labelpad=10)
ax.set_ylabel("v8.13.2 wall hits - v8.5.31 wall hits", labelpad=10)
ax.set_xlim(0, NUM_TEST_EPISODES + 1)
ax.set_xticks(range(0, NUM_TEST_EPISODES + 1, 10))
ax.grid(axis="x", linestyle="--", alpha=0.28)
ax.grid(axis="y", linestyle="-", alpha=0.18)
plt.show()

paired.sort_values("diff_v8.13.2_minus_v8.5.31").head(10)

In [ ]:
def hit_heatmap(events, run_name, value_column="from"):
    grid = np.zeros((5, 5), dtype=int)
    rows = events[events["model"] == run_name]
    if value_column == "intended":
        coords = zip(rows["intended_row"], rows["intended_col"])
    else:
        coords = zip(rows["from_row"], rows["from_col"])
    for row, col in coords:
        if 0 <= row < 5 and 0 <= col < 5:
            grid[int(row), int(col)] += 1
    return grid


fig, axes = plt.subplots(1, 2, figsize=(12, 5.8), dpi=140)
max_count = max(hit_heatmap(wall_hit_events, run["name"]).max() for run in RUNS)

for ax, run in zip(axes, RUNS):
    grid = hit_heatmap(wall_hit_events, run["name"])
    image = ax.imshow(grid, cmap="Blues", vmin=0, vmax=max_count)
    ax.set_title(f"{run['short_name']} wall-hit origins", loc="left", pad=12)
    ax.set_xlabel("Column", labelpad=10)
    ax.set_ylabel("Row", labelpad=10)
    ax.set_xticks(range(5))
    ax.set_yticks(range(5))
    for row in range(5):
        for col in range(5):
            value = grid[row, col]
            color = "white" if value > max_count * 0.45 else "#222222"
            ax.text(col, row, str(value), ha="center", va="center", color=color, fontsize=10)

fig.colorbar(image, ax=axes, shrink=0.82, label="Wall hits")
plt.show()

wall_hit_events.groupby(["short_name", "from_row", "from_col", "action_name", "kind"]).size().reset_index(name="wall_hits").sort_values("wall_hits", ascending=False).head(20)